In [ ]:
import numpy as np
import math
from data_prep import prep_and_load_data
from model import get_model
import constants as CONST
import pickle
import matplotlib.pyplot as plt
import cv2
import os
import copy

# Update the constants for training and testing directories
CONST.TRAIN_DIR = r'C:\Users\CERTAN\train'
CONST.TEST_DIR = r'C:\Users\CERTAN\test1'

# Load image data and labels from the specified directories
images, labels = prep_and_load_data()

def plotter(history_file):
    with open(history_file, 'rb') as file:
        history = pickle.load(file)

    plt.plot(history['accuracy'])
    plt.plot(history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['train', 'val'], loc='upper left')
    plt.savefig('accuracy.png')
    plt.show()

    plt.plot(history['loss'])
    plt.plot(history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['train', 'val'], loc='upper left')
    plt.savefig('loss.png')
    plt.show()

def process_image(directory, img_path):
    path = os.path.join(directory, img_path)
    image = cv2.imread(path)
    image_copy = copy.deepcopy(image)
    image = cv2.resize(image, (CONST.IMG_SIZE, CONST.IMG_SIZE))
    image_std = image.astype('float') / 255.0
    return image_copy, image_std

def video_write(model):
    fourcc = cv2.VideoWriter_fourcc(*'DIVX')
    out = cv2.VideoWriter("./prediction.mp4", fourcc, 1.0, (400, 400))
    val_map = {0: 'Buaya', 1: 'Alligator', 2: 'Komodo', 3: 'Cicak'}

    font = cv2.FONT_HERSHEY_SIMPLEX
    location = (20, 20)
    fontScale = 0.5
    fontColor = (255, 255, 255)
    lineType = 2

    DIR = CONST.TEST_DIR
    image_paths = os.listdir(DIR)
    image_paths = image_paths[:200]  # Limit to first 200 images
    count = 0
    for img_path in image_paths:
        image, image_std = process_image(DIR, img_path)
        image_std = image_std.reshape(-1, CONST.IMG_SIZE, CONST.IMG_SIZE, 3)
        pred = model.predict(image_std)  # Predict the class
        arg_max = np.argmax(pred, axis=1)
        max_val = np.max(pred, axis=1)
        s = f"{val_map[arg_max[0]]} - {max_val[0] * 100:.2f}%"
        cv2.putText(image, s, location, font, fontScale, fontColor, lineType)
        frame = cv2.resize(image, (400, 400))
        out.write(frame)
        count += 1
    out.release()

if __name__ == "__main__":
    images, labels = prep_and_load_data()
    
    # Calculate training size
    train_size = int(CONST.DATA_SIZE * CONST.SPLIT_RATIO)
    
    # Split the data for training and testing
    train_data = images[:train_size]
    train_labels = labels[:train_size]
    test_data = images[train_size:]
    test_labels = labels[train_size:]

    # Build and train the model
    model = get_model()
    history = model.fit(train_data, train_labels, batch_size=50, epochs=20, validation_data=(test_data, test_labels))
    
    # Save the trained model
    model.save('model.keras')

    # Save the training history
    with open('history.pickle', 'wb') as file:
        pickle.dump(history.history, file)

    # Plot accuracy and loss graphs
    plotter('history.pickle')
    
    # Write predictions to a video
    video_write(model)
